Pregunta 11. Agregación con filtro – Costo del marketing inútil
Enunciado: Sabiendo que a la agencia (canal TA/TO) se le paga USD 1.5 por cada reserva. ¿Cuánto dinero en total se ha pagado a la agencia por reservas que finalmente fueron canceladas (is_canceled = 1)? Escribe la consulta que calcule este monto de pérdida.

In [0]:
%sql
SELECT
    COUNT(*) AS reservas_canceladas,
    COUNT(*) * 1.5 AS perdida_total_usd
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE distribution_channel = 'TA/TO'
AND is_canceled = 1;

conclusion: Se calculó el gasto destinado a las agencias por las reservas canceladas del canal TA/TO. Este monto representa una pérdida económica que debe considerarse al renegociar los contratos de marketing.

Pregunta 12. DATE_DIFF – Costo de cancelar tarde
Enunciado: Cuando un huésped cancela con menos de 3 días de anticipación respecto a su llegada, el hotel pierde USD 120. Escribe una consulta que cuente cuántas reservas cumplen esta condición (is_canceled = 1 y diferencia entre llegada y cancelación menor a 3 días) y multiplica esa cantidad por 120 para ver la pérdida total.

In [0]:
%sql
WITH reservas AS (
    SELECT
        is_canceled,
        TO_DATE(
            CONCAT(
                arrival_date_year, '-',
                arrival_date_month, '-',
                arrival_date_day_of_month
            ),
            'yyyy-MMMM-d'
        ) AS arrival_date,

        COALESCE(
            TO_DATE(TRY_TO_TIMESTAMP(reservation_status_date, 'M/d/yy')),
            TO_DATE(TRY_TO_TIMESTAMP(reservation_status_date, 'yyyy-MM-dd'))
        ) AS fecha_cancelacion

    FROM cursodataacademy.proyecto_hotel.hotel_bookings
)
SELECT
    COUNT(*) AS cancelaciones_tardias,
    COUNT(*) * 120 AS perdida_total_usd
FROM reservas
WHERE is_canceled = 1
AND fecha_cancelacion IS NOT NULL
AND DATEDIFF(arrival_date, fecha_cancelacion) BETWEEN 0 AND 2;

conclusion: Se identificaron las reservas canceladas con menos de tres días de anticipación y se calculó el costo operativo de USD 120 por cada caso. Este resultado permite cuantificar las pérdidas ocasionadas por cancelaciones tardías.

Pregunta 13. LEFT JOIN – Relación de reservas con países
Enunciado: Imagina que tenemos una segunda tabla llamada tb_paises con las columnas cod_pais y nombre_pais. Une tu tabla de reservas (usando su columna country) con tb_paises mediante un LEFT JOIN. Selecciona si la reserva fue cancelada y el nombre_pais.

Paso 1: crear la tabla

In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.proyecto_hotel.tb_paises (
    cod_pais STRING,
    nombre_pais STRING
);

Paso 2: insertar registros de ejemplo

In [0]:
%sql
INSERT INTO cursodataacademy.proyecto_hotel.tb_paises
VALUES
('PRT', 'Portugal'),
('GBR', 'Reino Unido'),
('FRA', 'Francia'),
('ESP', 'España'),
('DEU', 'Alemania'),
('IRL', 'Irlanda'),
('ITA', 'Italia'),
('BEL', 'Bélgica'),
('BRA', 'Brasil'),
('NLD', 'Países Bajos');

Paso 3: realizar el LEFT JOIN

In [0]:
%sql
SELECT
    r.is_canceled,
    r.country,
    p.nombre_pais
FROM cursodataacademy.proyecto_hotel.hotel_bookings r
LEFT JOIN cursodataacademy.proyecto_hotel.tb_paises p
    ON r.country = p.cod_pais;

Conclusión: El LEFT JOIN permite relacionar las reservas con los países, conservando todos los registros de la tabla original. Asimismo, facilita identificar los países asociados a cada reserva y aquellos que no cuentan con correspondencia en la tabla auxiliar.

Pregunta 14. INNER JOIN – Países con mayor cantidad de cancelaciones
Enunciado: Usando el mismo JOIN del ejercicio anterior, averigua cuáles son los cinco países que tienen la mayor cantidad de reservas canceladas.

In [0]:
%sql
SELECT
    p.nombre_pais,
    COUNT(*) AS total_cancelaciones
FROM cursodataacademy.proyecto_hotel.hotel_bookings r
INNER JOIN cursodataacademy.proyecto_hotel.tb_paises p
    ON r.country = p.cod_pais
WHERE r.is_canceled = 1
GROUP BY p.nombre_pais
ORDER BY total_cancelaciones DESC
LIMIT 5;

Conclusión: Portugal registra la mayor cantidad de reservas canceladas, con 27,519 casos, seguido del Reino Unido con 2,453. Esto evidencia una concentración considerable de cancelaciones en Portugal dentro de los países incluidos en el análisis.

Pregunta 15. FULL OUTER JOIN – Relación de vendedores y reservas
Enunciado: Supongamos que Recursos Humanos tiene una tabla tb_vendedores (con id_vendedor y nombre_vendedor) y cruzamos las reservas para saber qué vendedor atendió qué reserva. Realiza un FULL OUTER JOIN entre las reservas y los vendedores. Selecciona el canal de distribución y el nombre del vendedor.

Paso 1:Crear la tabla de vendedores

In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.proyecto_hotel.tb_vendedores (
    id_vendedor INT,
    nombre_vendedor STRING
);

Paso 2:Insertar registros de vendedores

In [0]:
%sql
INSERT INTO cursodataacademy.proyecto_hotel.tb_vendedores
VALUES
(1, 'Carlos Mendoza'),
(2, 'Ana Torres'),
(3, 'Luis Ramírez'),
(4, 'María Fernández');

Paso 3:Crear una tabla auxiliar de reservas con vendedores

In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.proyecto_hotel.tb_reservas_vendedor AS

WITH reservas_numeradas AS (
    SELECT
        ROW_NUMBER() OVER (
            ORDER BY arrival_date_year,
                     arrival_date_month,
                     arrival_date_day_of_month,
                     lead_time
        ) AS id_reserva,
        distribution_channel
    FROM cursodataacademy.proyecto_hotel.hotel_bookings
)

SELECT
    id_reserva,
    distribution_channel,
    CASE
        WHEN pmod(id_reserva, 4) = 0 THEN CAST(NULL AS INT)
        WHEN pmod(id_reserva, 4) = 1 THEN 1
        WHEN pmod(id_reserva, 4) = 2 THEN 2
        ELSE 3
    END AS id_vendedor
FROM reservas_numeradas;

In [0]:
%sql
SELECT
    r.id_reserva,
    r.distribution_channel,
    v.id_vendedor,
    v.nombre_vendedor
FROM cursodataacademy.proyecto_hotel.tb_reservas_vendedor r
FULL OUTER JOIN cursodataacademy.proyecto_hotel.tb_vendedores v
    ON r.id_vendedor = v.id_vendedor;

Conclusión: El FULL OUTER JOIN permite identificar las reservas con vendedores asignados, las reservas sin atención y los vendedores sin reservas asociadas. Esto facilita evaluar la distribución de responsabilidades y detectar posibles problemas en la asignación comercial.

Pregunta 16. RIGHT JOIN – Relación de reservas y facturación
Enunciado: El departamento de contabilidad tiene una tabla de facturación (tb_facturas con id_reserva y monto_facturado). Usando un RIGHT JOIN desde las reservas hacia las facturas, garantiza que veamos todas las facturas emitidas, cruzadas con el tipo de hotel.

Paso 1:Crear una tabla auxiliar de reservas con identificador

In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.proyecto_hotel.tb_reservas_con_id AS
SELECT
    ROW_NUMBER() OVER (
        ORDER BY hotel,
                 arrival_date_year,
                 arrival_date_month,
                 arrival_date_day_of_month,
                 country,
                 lead_time
    ) AS id_reserva,
    hotel
FROM cursodataacademy.proyecto_hotel.hotel_bookings;

Paso 2:Crear la tabla de facturas

In [0]:
%sql
CREATE OR REPLACE TABLE cursodataacademy.proyecto_hotel.tb_facturas (
    id_reserva BIGINT,
    monto_facturado DECIMAL(10,2)
);

Paso 3:Insertar registros de facturación

In [0]:
%sql
INSERT INTO cursodataacademy.proyecto_hotel.tb_facturas
VALUES
(1, 250.00),
(2, 180.00),
(3, 320.00),
(999999, 150.00);

In [0]:
%sql
SELECT
    f.id_reserva,
    f.monto_facturado,
    r.hotel
FROM cursodataacademy.proyecto_hotel.tb_reservas_con_id r
RIGHT JOIN cursodataacademy.proyecto_hotel.tb_facturas f
    ON r.id_reserva = f.id_reserva;

Conclusión: El RIGHT JOIN permitió conservar las 4 facturas emitidas, identificando que 3 corresponden al City Hotel y 1 no tiene una reserva asociada. Esto demuestra la utilidad de esta operación para detectar facturas sin correspondencia en los registros de reservas.